# **1. Import Library**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report
import joblib

# **2. Memuat Dataset dari Hasil Clustering**
Memuat dataset hasil clustering dari file CSV ke dalam variabel DataFrame.

In [ ]:
df = pd.read_csv("data_clustering_inverse.csv")

In [ ]:
df.head()

,TransactionAmount,TransactionType,Location,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,CustomerAge_bin,TransactionAmount_bin,Target
0,14.09,Debit,San Diego,ATM,70.0,Doctor,81.0,1.0,5112.21,Tua,Kecil,1
1,376.24,Debit,Houston,ATM,68.0,Doctor,141.0,1.0,13758.91,Tua,Besar,0
2,126.29,Debit,Mesa,Online,19.0,Student,56.0,1.0,1122.35,Muda,Sedang,1
3,184.50,Debit,Raleigh,Online,26.0,Student,25.0,1.0,8569.06,Muda,Sedang,1
4,92.15,Debit,Oklahoma City,ATM,18.0,Student,172.0,1.0,781.68,Muda,Kecil,1


**Feature Encoding: One Hot Encoding**

In [ ]:
categorical_cols = list(df.select_dtypes(include=['object']).columns)

df_encoded = pd.get_dummies(
    df,
    columns = categorical_cols,
    drop_first = True
)
df_encoded.head()

,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,Target,TransactionType_Debit,Location_Atlanta,Location_Austin,Location_Baltimore,...,Location_Washington,Channel_Branch,Channel_Online,CustomerOccupation_Engineer,CustomerOccupation_Retired,CustomerOccupation_Student,CustomerAge_bin_Muda,CustomerAge_bin_Tua,TransactionAmount_bin_Kecil,TransactionAmount_bin_Sedang
0,14.09,70.0,81.0,1.0,5112.21,1,True,False,False,False,...,False,False,False,False,False,False,False,True,True,False
1,376.24,68.0,141.0,1.0,13758.91,0,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,126.29,19.0,56.0,1.0,1122.35,1,True,False,False,False,...,False,False,True,False,False,True,True,False,False,True
3,184.50,26.0,25.0,1.0,8569.06,1,True,False,False,False,...,False,False,True,False,False,True,True,False,False,True
4,92.15,18.0,172.0,1.0,781.68,1,True,False,False,False,...,False,False,False,False,False,True,True,False,True,False


# **3. Data Splitting**

In [ ]:
X = df_encoded.drop('Target', axis=1)
y = df_encoded['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

print("Jumlah data total: ",len(X))
print("Jumlah data latih: ",len(X_train))
print("Jumlah data test: ",len(X_test))

Jumlah data total:  1945
Jumlah data latih:  1556
Jumlah data test:  389


# **4. Membangun Model Klasifikasi**

In [ ]:
decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [ ]:
joblib.dump(decision_tree_model, 'decision_tree_model.h5')

['decision_tree_model.h5']

In [ ]:
new_model = RandomForestClassifier(random_state=42)
new_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [ ]:
y_pred_dt = decision_tree_model.predict(X_test)
y_pred_new = new_model.predict(X_test)

# Tampilkan classification_report untuk Decision Tree
print("Decision Tree Performance")
print(classification_report(y_test, y_pred_dt))

print("="*50)

# Tampilkan classification_report untuk New Model
print("New Model Performance")
print(classification_report(y_test, y_pred_new))

Decision Tree Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389

New Model Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389



In [ ]:
joblib.dump(new_model, 'explore_RandomForest_classification.h5')

['explore_RandomForest_classification.h5']

Hyperparameter Tuning Model

Pilih salah satu algoritma yang ingin Anda tuning

In [ ]:
# Tentukan Hyperparameter yang akan di-tuning
params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
}

new_model_tuned = GridSearchCV(
    estimator = RandomForestClassifier(random_state=42),
    param_grid = params,
    cv = 5,
    scoring = 'accuracy'
)

new_model_tuned.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
             param_grid={'max_depth': [None, 5, 10],
                         'min_samples_split': [2, 5],
                         'n_estimators': [50, 100, 200]},
             scoring='accuracy')

In [ ]:
# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada algoritma yang sudah dituning.

# Buat prediksi pada 'X_test' Gunakan model yang sudah di-tuning
y_pred_tuning = new_model_tuned.predict(X_test)

# Tampilkan classification_report untuk model yang sudah di-tuning
print("Tuned Model Performance")
print(classification_report(y_test, y_pred_tuning))

Tuned Model Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389



In [ ]:
# Menyimpan Model hasil tuning
joblib.dump(new_model_tuned, 'tuning_classification.h5')

['tuning_classification.h5']